# GK-2A TA/HM 피처 엔지니어링 최적화

기존 CatBoost + residual LSTM을 고정 기준으로 두고, 규정상 허용되는 피처군을 하나씩 추가합니다. 2022~2024 rolling OOF로 선택하고, 중복 그룹과 상관 피처를 제거한 뒤 2025는 마지막 확인에만 사용합니다. TA/HM은 독립적으로 선택됩니다.

In [ ]:
# 1. Drive 안전 마운트
from pathlib import Path
from google.colab import drive
import os

MOUNT_ROOT = Path('/content/drive')
if not os.path.ismount(MOUNT_ROOT):
    if MOUNT_ROOT.exists() and any(MOUNT_ROOT.iterdir()):
        MOUNT_ROOT = Path('/content/gdrive')
    drive.mount(str(MOUNT_ROOT), force_remount=False)
MYDRIVE = MOUNT_ROOT / 'MyDrive'
print('MyDrive:', MYDRIVE)

In [ ]:
# 2. 전용 브랜치 동기화
import subprocess, shutil, sys
REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_BRANCH = 'agent/feature-engineering-optimizer'
REPO_DIR = Path('/content/SME_DATA_feature_optimizer')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], cwd=REPO_DIR, check=True)
else:
    if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
print(REPO_DIR)

In [ ]:
# 3. 의존성
%pip install -q catboost==1.2.8 xarray h5netcdf
import pandas as pd, json
print('python:', sys.version)

In [ ]:
# 4. 데이터와 기존 baseline 결과 자동 탐색
DATA_ROOT = MYDRIVE / 'SME_DATA' / 'processed_station_features'
SHORTTERM_CSV = DATA_ROOT / 'shortterm_12to14_data' / 'incremental_12to14_tables' / 'shortterm_long_2019to2025.csv'
BASELINE_ROOT = DATA_ROOT / 'model_experiments_19to25'
matches = list(BASELINE_ROOT.rglob('step0_current_baseline_TA_oof.csv'))
if not SHORTTERM_CSV.exists(): raise FileNotFoundError(SHORTTERM_CSV)
if not matches:
    raise FileNotFoundError(f'baseline OOF가 없습니다: {BASELINE_ROOT}')
matches = sorted(matches, key=lambda p: ('seed42' not in str(p).lower(), len(str(p))))
BASELINE_DIR = matches[0].parent
OUTPUT_DIR = BASELINE_ROOT / 'feature_engineering_optimizer'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('shortterm:', SHORTTERM_CSV)
print('baseline :', BASELINE_DIR)
print('output   :', OUTPUT_DIR)

## 선택 사항: 14시 공간 패치 추출

기본값은 꺼져 있습니다. 원본 NC가 Drive의 `shortterm_12to14_data/raw_gk2a`에 있을 때만 `RUN_SPATIAL=True`로 바꾸세요. 완료된 날짜는 재사용하므로 중간에 끊겨도 다시 실행할 수 있습니다.

In [ ]:
# 5. 공간 피처 1차 실험: 14시 thermal/WV 10채널, 5km/15km 패치
RUN_SPATIAL = False
RAW_ROOT = DATA_ROOT / 'shortterm_12to14_data'
SPATIAL_DIR = RAW_ROOT / 'spatial_patch_features_1400'
if RUN_SPATIAL:
    for year in range(2019, 2026):
        cmd = [
            sys.executable, '-u', str(REPO_DIR / 'scripts' / 'extract_shortterm_spatial_features.py'),
            '--input-root', str(RAW_ROOT), '--output-dir', str(SPATIAL_DIR),
            '--start', f'{year}-08-24', '--end', f'{year}-08-30', '--time-kst', '14:00',
            '--config', str(REPO_DIR / 'configs' / 'data.yaml'),
            '--station-list', str(REPO_DIR / 'data' / 'metadata' / 'station_list.csv'),
        ]
        subprocess.run(cmd, cwd=REPO_DIR, check=True)
SPATIAL_CSV = SPATIAL_DIR / 'spatial_features_combined.csv'
else:
    SPATIAL_CSV = None
print('spatial:', SPATIAL_CSV)

In [ ]:
# 6. Ridge 잔차 피처 최적화: 추가 -> 그룹 제거 -> 상관 중복 제거
SCRIPT = REPO_DIR / 'scripts' / 'experiment_feature_engineering_optimizer.py'
RIDGE_OUT = OUTPUT_DIR / ('ridge_with_spatial' if SPATIAL_CSV else 'ridge_table_only')
cmd = [
    sys.executable, '-u', str(SCRIPT),
    '--shortterm-long-csv', str(SHORTTERM_CSV), '--output-dir', str(RIDGE_OUT),
    '--targets', 'TA', 'HM', '--selection-years', '2022', '2023', '2024',
    '--report-year', '2025', '--estimator', 'ridge', '--selection-mode', 'baseline_residual',
    '--baseline-ta-oof', str(BASELINE_DIR / 'step0_current_baseline_TA_oof.csv'),
    '--baseline-ta-test', str(BASELINE_DIR / 'step0_current_baseline_TA_test.csv'),
    '--baseline-hm-oof', str(BASELINE_DIR / 'step0_current_baseline_HM_oof.csv'),
    '--baseline-hm-test', str(BASELINE_DIR / 'step0_current_baseline_HM_test.csv'),
    '--threads', '4', '--max-prune-steps', '12',
]
if SPATIAL_CSV: cmd += ['--spatial-features-csv', str(SPATIAL_CSV)]
print('$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 7. 결과 확인
summary = json.loads((RIDGE_OUT / 'feature_engineering_summary.json').read_text())
print(json.dumps(summary['targets'].get('competition_proxy', {}), ensure_ascii=False, indent=2))
display(pd.read_csv(RIDGE_OUT / 'forward_addition_trials.csv')[
    ['target','stage','candidate','feature_count','pooled_RMSE','delta_RMSE','accepted']
])
display(pd.read_csv(RIDGE_OUT / 'correlation_pruning_trials.csv')[
    ['target','prune_step','candidate','feature_count','pooled_RMSE','delta_RMSE','accepted']
])
print('결과 저장:', RIDGE_OUT)

## 선택 사항: 기존 LSTM 시드 3개 재검증

Drive에 seed 42~44의 `step0_current_baseline_*` 결과가 모두 있으면, 선택된 23/3개 피처를 고정한 상태에서 시드별 잔차 Ridge만 재학습합니다. 피처를 다시 고르지 않으므로 선택 편향을 늘리지 않습니다.

In [ ]:
# 8. 다중 시드 확인
import re
baseline_dirs = sorted({p.parent for p in BASELINE_ROOT.rglob('step0_current_baseline_TA_oof.csv')})
complete_dirs = [d for d in baseline_dirs if all((d / f'step0_current_baseline_{t}_{s}.csv').exists() for t in ['TA','HM'] for s in ['oof','test'])]
if len(complete_dirs) >= 3:
    chosen = complete_dirs[:3]
    labels = []
    for i, d in enumerate(chosen, 1):
        match = re.search(r'seed(\d+)', str(d).lower())
        labels.append(match.group(1) if match else str(41 + i))
    MULTI_OUT = OUTPUT_DIR / 'multiseed_confirmation'
    cmd = [
        sys.executable, '-u', str(REPO_DIR / 'scripts' / 'confirm_feature_optimizer_multiseed.py'),
        '--shortterm-long-csv', str(SHORTTERM_CSV),
        '--optimizer-summary', str(RIDGE_OUT / 'feature_engineering_summary.json'),
        '--baseline-dirs', *map(str, chosen), '--seed-labels', *labels,
        '--output-dir', str(MULTI_OUT), '--selection-years', '2022', '2023', '2024',
        '--report-year', '2025',
    ]
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(pd.read_csv(MULTI_OUT / 'multiseed_competition_scores.csv'))
    display(pd.read_csv(MULTI_OUT / 'multiseed_summary.csv'))
else:
    print('완전한 baseline seed 폴더가 3개 미만이라 다중 시드 셀을 건너뜁니다:', complete_dirs)

## 해석 규칙

- `accepted=True`인 추가군만 다음 단계에 전달됩니다.
- 그룹 제거 후 신규군이 사라졌다면 그 신규정보는 기존 피처와 중복되거나 일반화되지 않은 것입니다.
- 상관 제거는 학습연도에서 |r|≥0.985인 군만 대상으로 하며, OOF 저하가 허용오차 이내일 때만 삭제합니다.
- HM의 engineered weight가 0이면 기존 HM 모델을 그대로 유지하는 것이 결론입니다.
- 2025 결과는 확인용이며 피처 선택에는 사용되지 않습니다.